# 02 - Data Preprocessing

> Research prototype only. The dataset is simulated and labels are experimental proxies. No result here validates field deployment or early-warning lead time.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_config, load_raw_data
from src.data_validation import validate_target
from src.feature_engineering import add_engineered_features, select_feature_columns, verify_flow_conventions

config = load_config()
frame = load_raw_data(config=config)
target = validate_target(frame, config)

## Cleaning decisions

There are no missing values in the raw file; five exact duplicate rows are retained because they may represent genuine steady-state simulator samples. Constant columns and target-derived columns are excluded from model inputs.

In [ ]:
print('Missing values:', int(frame.isna().sum().sum()))
print('Duplicate rows:', frame.duplicated().sum())
featured = add_engineered_features(frame, config)
print(verify_flow_conventions(featured))
columns = select_feature_columns(featured, config)
print('Selected features:', columns)

## Scaling and imputation

The `ColumnTransformer` below is fitted on training data only. Robust scaling is used because genuine transient kick responses should not be treated as errors.

In [ ]:
from src.preprocessing import build_preprocessor, split_data, make_validation_split
split = split_data(featured[columns], target.labels, config)
preprocessor = build_preprocessor(columns, config)
X_train_t = preprocessor.fit_transform(split.X_train)
print('Transformed training matrix:', X_train_t.shape)
fit, val, y_fit, y_val = make_validation_split(split.X_train, split.y_train, split.train_groups, config)
print('Fit/validation rows:', len(fit), len(val))

## Group-aware split

Contiguous 25-row blocks remain intact and are stratified by the proxy class, reducing leakage from adjacent near-duplicate measurements. The test set is not used for any fitting or tuning.

In [ ]:
print('Train/test rows:', len(split.X_train), len(split.X_test))
print('Positive fraction:', split.y_train.mean(), split.y_test.mean())
split.X_train.assign(target=split.y_train.to_numpy()).head()